# Notebook 01: The Agent Loop

We move from single-turn LLM calls to agents that plan, act, observe, and adapt.

**Expected time:** ~45 minutes  
**Prerequisites:** Notebook 00

```mermaid
graph TD
    User([User Input]) --> Thought[Thought: Reason about next steps]
    Thought --> Action[Action: Call a specific Tool]
    Action --> Observation[Observation: Get Tool output]
    Observation --> Thought
    Thought --> Answer[Final Answer: Return text output to user]
```


## What You'll Learn

1. The difference between a chatbot and an agent
2. The ReAct pattern: Thought → Action → Observation → repeat
3. Use a pre-built agent from the helper module
4. Define tools and connect them to an agent
5. Recognize and fix common agent failures
6. Peek under the hood at a simplified loop implementation


## Setup

Run the cell below to install packages and import helpers. If you skipped Notebook 00, you'll also need an API key (see Notebook 00 for instructions).

This notebook supports **Groq** and **OpenRouter**. The default OpenRouter model is `nvidia/nemotron-3-ultra-550b-a55b:free`. Set your provider's mock flag to `True` if you have no API key.

In [1]:
!pip install openai==1.68.2 groq==0.18.0

from agent_helpers import ReactAgent, make_tool, mock_llm, mock_search, tools_to_openai_native, safe_calc
import getpass
import inspect
import json

import os
from pathlib import Path
from openai import OpenAI

# --- Optional .env file loading (no extra pip installs needed) ---
env_path = Path.cwd() / ".env"
if env_path.exists():
    with open(env_path) as f:
        for line in f:
            line = line.strip()
            if line and not line.startswith("#") and "=" in line:
                k, v = line.split("=", 1)
                os.environ.setdefault(k.strip(), v.strip())
    print("Loaded .env file.")
else:
    print("No .env file found — you will be prompted for API keys.")
    print("  Tip: cp .env.example .env  and fill in your keys to skip prompts.")

PROVIDER = "openrouter"  # Choose: groq, openrouter

MOCK = {
    "groq":       True,
    "openrouter": False,  # Set True to skip real API call
}

# Environment variable names for auto-loading keys (set in .env)
KEY_ENV_VAR = {
    "groq":       "GROQ_API_KEY",
    "openrouter": "OPENROUTER_API_KEY",
}

CONFIG = {
    "groq": {
        "name": "Groq",
        "model": "llama-3.3-70b-versatile",
        "key_url": "https://console.groq.com/keys",
        "base_url": "https://api.groq.com/openai/v1",
    },
    "openrouter": {
        "name": "OpenRouter",
        "model": "deepseek/deepseek-v4-flash",
        "key_url": "https://openrouter.ai/keys",
        "base_url": "https://openrouter.ai/api/v1",
    },
}

cfg = CONFIG[PROVIDER]
use_mock = MOCK[PROVIDER]

if use_mock:
    print(f"[{cfg['name']}] Mock mode enabled.")
    client = None
else:
    api_key = os.environ.get(KEY_ENV_VAR[PROVIDER]) or getpass.getpass(f"Enter your {cfg['name']} API key: ")
    client = OpenAI(api_key=api_key, base_url=cfg["base_url"])
    print(f"[{cfg['name']}] API key configured.")


Loaded .env file.
[OpenRouter] API key configured.


### Model Wrapper

The agent loop needs a function that sends messages to the LLM and returns the response text. This wrapper adapts the provider SDK to the simple interface `ReactAgent` expects.

In [2]:
def make_model(client, model_name=None, system_prompt_tools=None):
    """Wrap provider client as a callable for ReactAgent.

    Uses native function calling API (tools param) for robust tool use.
    """
    if model_name is None:
        model_name = cfg["model"]

    native_tools = None
    system_prompt = None

    if system_prompt_tools:
        native_tools = tools_to_openai_native(system_prompt_tools)
        system_prompt = (
            "You have access to tools. Use them when needed to answer user questions."
            " If you can answer without a tool, respond directly."
        )

    def model_fn(messages):
        msgs = messages
        if system_prompt and not any(m.get("role") == "system" for m in msgs):
            msgs = [{"role": "system", "content": system_prompt}] + msgs

        kwargs = {"model": model_name, "messages": msgs}
        if native_tools:
            kwargs["tools"] = native_tools

        response = client.chat.completions.create(**kwargs)
        msg = response.choices[0].message

        if msg.tool_calls:
            tc = msg.tool_calls[0]
            return {
                "type": "tool_call",
                "name": tc.function.name,
                "args": json.loads(tc.function.arguments),
                "tool_call_id": tc.id,
            }
        return {"type": "text", "content": msg.content}

    return model_fn


## Chatbot vs. Agent

A **chatbot** answers once. An **agent** loops: it can call tools, see results, and decide what to do next.

Let's see the difference side by side with the same task:
> *"What is (15 * 3) + 42 / 2?"*


### Chatbot (Single Turn)


In [3]:
if not use_mock and client is not None:
    model = make_model(client, cfg["model"])
    reply = model([{"role": "user", "content": "What is (15 * 3) + 42 / 2?"}])
    print("Chatbot response:")
    print(reply)
    print("\nOne answer. No follow-up. No tool use.")
else:
    print("Mock — chatbot would respond with the computed answer.")
    print("But a real LLM might calculate wrong without a tool.")


Chatbot response:
{'type': 'text', 'content': 'The result of (15 * 3) + 42 / 2 is 66.'}

One answer. No follow-up. No tool use.


### Agent (With Tool)


In [4]:
# Define a calculator tool (uses safe_calc — no eval security risk)
def calculator(expr):
    return safe_calc(expr)

tools = [make_tool("calculator", "Evaluate arithmetic expressions", calculator)]

if not use_mock and client is not None:
    model = make_model(client, cfg["model"], system_prompt_tools=tools)
    agent = ReactAgent(model=model, tools=tools)
    result = agent.run("What is (15 * 3) + 42 / 2?")
    print("Agent response:")
    print(result)
    print("\nThe agent called the calculator tool internally to get the exact answer.")
else:
    mock_model = mock_llm("TOOL: calculator, args: {\"expr\": \"(15*3)+(42/2)\"}")
    agent = ReactAgent(model=mock_model, tools=tools)
    result = agent.run("What is (15 * 3) + 42 / 2?")
    print("Agent response (mock):")
    print(result)


Agent response:
The answer is **66**. 

Here's the breakdown:
- \(15 \times 3 = 45\)
- \(42 / 2 = 21\)
- \(45 + 21 = 66\)

The agent called the calculator tool internally to get the exact answer.


## Tool Definition Pattern

Tools are the agent's hands. Each tool has a name, description, and function.

```python
tools = [
    make_tool("name", "what it does", function),
    make_tool("another", "description", another_function),
]
```

Let's build a multi-tool agent that decides *which* tool to call based on the question.

In [5]:
def calculator(expr):
    return f"Result: {safe_calc(expr)}"

def get_weather(city):
    """Mock weather lookup."""
    data = {
        "Chisinau": "18°C, partly cloudy",
        "Copenhagen": "12°C, light rain",
        "Athens": "28°C, sunny",
        "Vienna": "15°C, overcast",
    }
    return data.get(city, f"Weather data not available for {city}")

def read_file(path):
    """Mock file reader."""
    files = {
        "notes.txt": "Agentic AI systems use a loop: plan, act, observe, adapt.",
        "todo.txt": "1. Finish agent loop notebook\n2. Start PBL project",
    }
    return files.get(path, f"File not found: {path}")

tools = [
    make_tool("calculator", "Evaluate math expressions", calculator),
    make_tool("get_weather", "Get weather for a city", get_weather),
    make_tool("read_file", "Read contents of a file", read_file),
]

print("Tools defined:")
for t in tools:
    print(f"  - {t['name']}: {t['description']}")


Tools defined:
  - calculator: Evaluate math expressions
  - get_weather: Get weather for a city
  - read_file: Read contents of a file


### Multi-Tool Agent in Action

Give the agent different questions and watch it pick the right tool.

In [6]:
if not use_mock and client is not None:
    model = make_model(client, cfg["model"], system_prompt_tools=tools)
    agent = ReactAgent(model=model, tools=tools)

    questions = [
        "What is 2 + 2?",
        "What is the weather in Chisinau?",
        "Read my notes.txt file",
    ]

    for q in questions:
        print(f">>> {q}")
        print(agent.run(q))
        print()
else:
    print("Mock mode — multi-tool agent responses would look like:")
    print('  >>> What is 2 + 2?'); print("  4")
    print('  >>> What is the weather in Chisinau?'); print("  18°C, partly cloudy")
    print('  >>> Read my notes.txt'); print("  Agentic AI systems use a loop: plan, act, observe, adapt.")


>>> What is 2 + 2?
2 + 2 = 4.

>>> What is the weather in Chisinau?
Here are the answers to your questions:

1. **2 + 2 = 4**
2. **Weather in Chișinău** 🌤️: It's currently **18°C** with **partly cloudy** skies.

>>> Read my notes.txt file
Here are the answers to your questions:

1. **2 + 2 = 4**
2. **Weather in Chisinau**: 18°C, partly cloudy ☁️
3. **Contents of notes.txt**: 
   > *"Agentic AI systems use a loop: plan, act, observe, adapt."*



## Peek Under the Hood

Here's a simplified version of how `ReactAgent` works internally. You don't need to write this — the helper module handles it — but reading it helps you understand the loop.

In [7]:
class SimpleReactAgent:
    """Simplified agent loop (15 lines). Read, don't type."""
    def __init__(self, model, tools, max_steps=5):
        self.model = model
        self.tools = {t["name"]: t for t in tools}
        self.history = []
        self.max_steps = max_steps

    def run(self, user_input):
        for step in range(self.max_steps):
            response = self.model(self.history + [{"role": "user", "content": user_input}])
            if "TOOL:" in response:
                name, args = self._parse(response)
                result = self.tools[name]["fn"](**args)
                self.history.append({"role": "tool", "content": str(result)})
            else:
                return response
        return "Max steps reached"

    def _parse(self, response):
        parts = response.replace("TOOL:", "").strip().split(", args:")
        name = parts[0].strip()
        args = json.loads(parts[1]) if len(parts) > 1 else {}
        return name, args

print("Simplified agent loop — 15 lines.")
print("The full version in agent_helpers.py adds error handling and better parsing.")


Simplified agent loop — 15 lines.
The full version in agent_helpers.py adds error handling and better parsing.


## What Breaks?

Agents fail in predictable ways. Let's break things on purpose.


### Break 1: Infinite Loop

Without a `max_steps` limit, an agent that never produces a final answer runs forever.

In [8]:
def stuck_model(messages):
    """Model that never gives a final answer — always calls a tool."""
    return "TOOL: calculator, args: {\"expr\": \"1+1\"}"

stuck_agent = ReactAgent(
    model=stuck_model,
    tools=[make_tool("calculator", "calc", lambda x: "2")],
    max_steps=3,  # Low limit so we see the stop quickly
)
result = stuck_agent.run("keep going")
print("Result after hitting max_steps=3:")
print(result)
print("\n☝️ The max_steps parameter is the safety net. Always set it.")


Result after hitting max_steps=3:
Max steps reached

☝️ The max_steps parameter is the safety net. Always set it.


### Break 2: Bad Tool JSON

The LLM sometimes generates malformed tool calls. The agent needs to handle this gracefully.

In [9]:
def bad_json_model(messages):
    """Model that returns unparseable JSON arguments."""
    return "TOOL: calculator, args: {invalid json here}"

try:
    agent = ReactAgent(
        model=bad_json_model,
        tools=[make_tool("calculator", "calc", lambda x: "2")],
    )
    agent.run("crash me")
except json.JSONDecodeError as e:
    print(f"Error: Malformed JSON from model — {e}")
    print("\nFix: Wrap json.loads() in try/except. In agent_helpers.py, this is already handled.")


### Break 3: Tool Not Found

The LLM requests a tool that doesn't exist. The agent needs to tell the LLM what's available.

In [10]:
def wrong_tool_model(messages):
    return "TOOL: send_email, args: {}"

agent = ReactAgent(
    model=wrong_tool_model,
    tools=[make_tool("calculator", "calc", lambda x: "2")],
)
result = agent.run("do something")
print("Agent response:")
print(result)
print("\nThe agent included the available tools in the error message. The LLM can try again with a valid tool.")


Agent response:
Max steps reached

The agent included the available tools in the error message. The LLM can try again with a valid tool.


## Connection to Agentic AI

The loop is what makes AI *agentic*. Without it, you have a chatbot — a one-shot answer generator. With it, the system can pursue multi-step goals, recover from failures, and use the real world (via tools).

This maps directly to **Complexity Ladder Level 1-2**:
- **Level 1:** One LLM call + one external tool (you did this with the calculator)
- **Level 2:** A loop with max_steps (you did this with the ReactAgent)

Every project in this course will be built on this loop.

### PBL Reflection


In [11]:
print("Think about your PBL project:")
print("  1. What task could your project automate with a loop?")
print("  2. What tools would your agent need?")
print("  3. Where might the loop get stuck? How would you protect it?")
print("\nWrite your thoughts in your project journal.")


Think about your PBL project:
  1. What task could your project automate with a loop?
  2. What tools would your agent need?
  3. Where might the loop get stuck? How would you protect it?

Write your thoughts in your project journal.


## Exercises


### Bronze — Run and Observe

Run the cell below. It creates an agent with a custom tool and runs 3 different prompts. Change the prompts and observe how the agent responds.

In [12]:
def search_web(query):
    """Mock web search."""
    results = {
        "agentic ai": "Agentic AI refers to systems that can pursue goals autonomously.",
        "python": "Python is a programming language widely used for AI and data science.",
        "PBL": "Problem-Based Learning: students learn by solving open-ended problems.",
    }
    return results.get(query.lower(), f"Search results for: {query}")

search_tools = [make_tool("search", "Search the web", search_web)]

if not use_mock and client is not None:
    model = make_model(client, cfg["model"], system_prompt_tools=search_tools)
    agent = ReactAgent(model=model, tools=search_tools)
else:
    mock = mock_llm("TOOL: search, args: {\"query\": \"default\"}")
    agent = ReactAgent(model=mock, tools=search_tools)

for question in ["What is agentic AI?", "Tell me about Python", "What is PBL?"]:
    print(f">>> {question}")
    print(agent.run(question))
    print()


>>> What is agentic AI?
Max steps reached

>>> Tell me about Python
## Python Programming Language

**Python** is a high-level, general-purpose programming language that has become one of the most popular languages in the world. Here's a comprehensive overview:

### Key Facts
- **Creator**: Guido van Rossum
- **First released**: 1991
- **Philosophy**: Emphasizes code readability and simplicity (the "Zen of Python")
- **Paradigm**: Supports multiple paradigms — procedural, object-oriented, and functional programming

### Core Features
1. **Readable & Clean Syntax** — Uses indentation instead of curly braces, making code easy to read and write
2. **Interpreted Language** — Code runs line by line, no compilation step needed
3. **Dynamically Typed** — No need to declare variable types explicitly
4. **Memory Management** — Automatic garbage collection (no manual memory handling)
5. **Extensive Standard Library** — "Batteries included" philosophy with modules for everything from file I/O to 

### Silver — Add Your Own Tool

Add a new tool to the agent. Write a function and register it with `make_tool`. Make it domain-relevant (e.g., a biotech lookup for Alejandro, a risk calculator for Jošt).

In [13]:
# --- Write your tool function ---
def my_custom_tool(param):
    """Describe what your tool does."""
    # Your logic here
    return f"Processed: {param}"
# ---------------------------------

custom_tools = [
    make_tool("calculator", "Evaluate math expressions", calculator),
    make_tool("my_tool", "Description of your tool", my_custom_tool),
]

if not use_mock and client is not None:
    model = make_model(client, cfg["model"], system_prompt_tools=custom_tools)
    agent = ReactAgent(model=model, tools=custom_tools)
    result = agent.run("Test my custom tool")
    print("Agent response:")
    print(result)
else:
    print(f"Tool registered: my_custom_tool")
    print("In real mode, the agent would call your tool.")


Agent response:
Your custom tool has been tested successfully! Here's what happened:

- **Tool invoked:** `my_tool`
- **Input parameter:** `"Hello, this is a test!"`
- **Output:** `"Processed: Hello, this is a test!"`

The tool received the input, processed it, and returned the expected response. Everything is working as intended! Let me know if you'd like to test it further or modify it.


### Gold — Implement ReactAgent From Scratch

Write your own agent loop without importing from `agent_helpers`. It must handle: max_steps, tool parsing, tool execution, and graceful failure.

In [14]:
# --- Implement your own ReactAgent ---
class MyAgent:
    def __init__(self, model, tools, max_steps=5):
        self.model = model
        self.tools = {t["name"]: t for t in tools}
        self.history = []
        self.max_steps = max_steps

    def run(self, user_input):
        # Implement the agent loop
        # Steps: for each step, call self.model(), check if it wants a tool,
        # execute tool, add observation to history, or return final answer
        pass  # Replace with your implementation
# --------------------------------------

# Test your implementation
# Note: 'tools' refers to the multi-tool list (calculator, weather, file) defined above.
# Define your own tools list here if you want to test with different tools.
if not use_mock and client is not None:
    model = make_model(client, cfg["model"])
    my_agent = MyAgent(model=model, tools=tools)
    result = my_agent.run("What is 2 + 2?")
    print("MyAgent result:", result)
else:
    print("Implement the run method, then test with real API.")
    print("Compare your implementation with agent_helpers.ReactAgent.")


MyAgent result: None


## Next Steps

You've built an agent loop. You understand the core pattern: model decides → tool executes → agent observes → repeat.

**Next notebook:** `02_memory_and_rag.ipynb` — "The Agent's Mind"

> "What happens when the agent needs to remember what it learned 20 turns ago?"
